In [7]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('FRED_API_KEY')

if api_key:
    print(f"✓ API key loaded successfully from .env")
    print(f"✓ Key starts with: {api_key[:8]}...")
else:
    print("✗ Key not found")

✓ API key loaded successfully from .env
✓ Key starts with: 3e0c1abb...


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('FRED_API_KEY')

if api_key:
    print(f"✓ API key loaded successfully from .env")
    print(f"✓ Key starts with: {api_key[:8]}...")
else:
    print("✗ Key not found")

✓ API key loaded successfully from .env
✓ Key starts with: 3e0c1abb...


In [9]:
import os
print(os.getcwd())

/Users/mark/Desktop/Coding/608/fed_project


In [10]:
import pandas as pd
import numpy as np
import altair as alt
from fredapi import Fred
from dotenv import load_dotenv
import os

load_dotenv()
fred = Fred(api_key=os.getenv('FRED_API_KEY'))

# ── PULL ALL DATA ──────────────────────────────────────────────
start = '2000-01-01'

fedfunds = fred.get_series('FEDFUNDS', observation_start=start).resample('MS').mean()
unrate   = fred.get_series('UNRATE', observation_start=start).resample('MS').mean()
cpi      = fred.get_series('CPIAUCSL', observation_start=start).resample('MS').mean()
housing  = fred.get_series('CSUSHPINSA', observation_start=start).resample('MS').last().ffill()
nasdaq   = fred.get_series('NASDAQCOM', observation_start=start).resample('MS').last().ffill()
wages    = fred.get_series('CES0500000003', observation_start=start).resample('MS').mean()

# ── BUILD MASTER DATAFRAME ─────────────────────────────────────
df = pd.DataFrame({'fedfunds': fedfunds, 'unrate': unrate, 'cpi': cpi})
df['inflation'] = df['cpi'].pct_change(12) * 100
df['wage_growth'] = wages.pct_change(12) * 100
df['real_wage'] = df['wage_growth'] - df['inflation']
df['cs_norm'] = (housing / housing.dropna().iloc[0]) * 100
df['nasdaq_norm'] = (nasdaq.reindex(df.index) / nasdaq.reindex(df.index).dropna().iloc[0]) * 100
df = df.reset_index().rename(columns={'index': 'date'})
df['date'] = pd.to_datetime(df['date'])

# Fed Chair eras
def get_chair(date):
    if date < pd.Timestamp('2006-02-01'): return 'Greenspan'
    elif date < pd.Timestamp('2014-02-01'): return 'Bernanke'
    elif date < pd.Timestamp('2018-02-01'): return 'Yellen'
    else: return 'Powell'

df['chair'] = df['date'].apply(get_chair)

# Save
df.to_csv('data/fed_mandate_master.csv', index=False)

print(f"✓ Data loaded: {len(df)} rows")
print(f"✓ Date range: {df['date'].min().strftime('%Y-%m')} to {df['date'].max().strftime('%Y-%m')}")
print(f"✓ Columns: {list(df.columns)}")
print(f"✓ Saved to data/fed_mandate_master.csv")

✓ Data loaded: 313 rows
✓ Date range: 2000-01 to 2026-01
✓ Columns: ['date', 'fedfunds', 'unrate', 'cpi', 'inflation', 'wage_growth', 'real_wage', 'cs_norm', 'nasdaq_norm', 'chair']
✓ Saved to data/fed_mandate_master.csv


In [11]:
# The Fed's Impossible Mandate
## A 25-Year Data Story

The Federal Reserve has two jobs: keep inflation low and keep unemployment low. 
On the surface that sounds reasonable. In practice, the tools required to do one 
often undermine the other — and 25 years of data shows the consequences.

This project argues three things:

1. **The mandate is structurally contradictory** — low inflation and low unemployment 
rarely coexist for long
2. **Each Fed rescue plants the next crisis** — cheap money transfers risk from one 
sector to the next, inflating larger bubbles each time
3. **The lag increases variance** — discretionary policy working on an 18-month delay 
is more fragile than rule-based frameworks, and political pressure makes it catastrophic

*Data source: Federal Reserve FRED API · Bureau of Labor Statistics · Case-Shiller · NASDAQ Composite*

SyntaxError: invalid character '—' (U+2014) (3150762087.py, line 6)

In [ ]:
## Part 1: The Contradiction

The Fed's mandate assumes inflation and unemployment can both be controlled 
simultaneously. The Phillips Curve — the relationship between the two — suggests 
otherwise: lower unemployment tends to push inflation up, and vice versa.

The scatter plot below shows 25 years of monthly data. Each dot is one month. 
**If the mandate were achievable, we'd see most dots clustered in the bottom-left 
— low inflation AND low unemployment together.** Look at where they actually land.

In [ ]:
# ── CHART 2 — THE CONTRADICTION ───────────────────────────────
df_chart2 = df.dropna(subset=['inflation', 'unrate'])

color_scale = alt.Scale(
    domain=['Greenspan', 'Bernanke', 'Yellen', 'Powell'],
    range=['#1f77b4', '#d62728', '#2ca02c', '#17becf']
)

scatter = alt.Chart(df_chart2).mark_circle(size=60, opacity=0.7).encode(
    x=alt.X('unrate:Q', title='Unemployment Rate (%)', scale=alt.Scale(domain=[3, 16])),
    y=alt.Y('inflation:Q', title='Inflation Rate (%)', scale=alt.Scale(domain=[-2, 12])),
    color=alt.Color('chair:N', scale=color_scale, title='Fed Chair'),
    tooltip=[
        alt.Tooltip('date:T', title='Date', format='%B %Y'),
        alt.Tooltip('unrate:Q', title='Unemployment %', format='.1f'),
        alt.Tooltip('inflation:Q', title='Inflation %', format='.1f'),
        alt.Tooltip('chair:N', title='Fed Chair')
    ]
)

target_line = alt.Chart(pd.DataFrame({'y': [2]})).mark_rule(
    color='red', strokeDash=[6, 4], opacity=0.6
).encode(y='y:Q')

target_label = alt.Chart(pd.DataFrame([{'x': 14, 'y': 2.4, 'text': '2% inflation target'}])
).mark_text(fontSize=11, color='red', opacity=0.7).encode(
    x='x:Q', y='y:Q', text='text:N'
)

chart2 = (scatter + target_line + target_label).properties(
    width=700, height=450,
    title=alt.TitleParams(
        text='Two Goals, One Lever — The Fed\'s Impossible Contradiction',
        subtitle='Each dot = one month. Low unemployment AND low inflation rarely appear together.',
        fontSize=15, subtitleFontSize=11
    )
).configure_axis(
    grid=True, gridOpacity=0.2,
    labelFontSize=11, titleFontSize=11
)

chart2

In [ ]:
# ── BUBBLE CYCLE DATA PREP ─────────────────────────────────────
df_bubble = df.dropna(subset=['cs_norm', 'nasdaq_norm']).copy()

# Normal trend for home prices (based on 2000-2003)
cs_pre = df_bubble[df_bubble['date'] < '2003-01-01']['cs_norm']
cs_slope = np.polyfit(np.arange(len(cs_pre)), cs_pre, 1)
df_bubble['cs_trend'] = np.polyval(cs_slope, np.arange(len(df_bubble)))

# Normal trend for NASDAQ (based on 2003-2006)
nasdaq_stable = df_bubble[
    (df_bubble['date'] >= '2003-01-01') &
    (df_bubble['date'] < '2006-01-01')
]['nasdaq_norm']
nasdaq_slope = np.polyfit(np.arange(len(nasdaq_stable)), nasdaq_stable, 1)
stable_start = df_bubble[df_bubble['date'] >= '2003-01-01'].index[0]
stable_pos = df_bubble.index.get_loc(stable_start)
df_bubble['nasdaq_trend'] = [np.polyval(nasdaq_slope, i - stable_pos) 
                              for i in range(len(df_bubble))]

print(f"✓ Bubble data ready: {len(df_bubble)} rows")

In [ ]:
## Part 2: The Bubble Cycle

When the economy crashes, the Fed cuts rates to fix it. Cheap money floods in. 
Prices inflate in whatever sector offers the best returns. Eventually the bubble pops. 
The Fed cuts rates again.

**The risk never disappears — it transfers.** Each cycle, the bubble is larger and 
more dangerous than the last. Click through the eras below to watch it happen.

In [ ]:
# ── EMBED BUBBLE VISUALIZATIONS ───────────────────────────────
from IPython.display import IFrame, display

display(IFrame('bubble_transfer_v2.html', width='100%', height=750))

In [ ]:
## Part 3: The Variance Problem

Even if the Fed makes the right call, rate changes take 12–18 months to affect 
the real economy. That delay doesn't just make the Fed late — it increases the 
variance of outcomes. A correct decision today can produce wildly different results 
depending on what happens in those 18 months.

This is what makes **discretionary policy fragile**. A rule-based framework like 
the Taylor Rule would have raised rates automatically in 2021 — no human judgment, 
no "transitory" call, no delay.

In [ ]:
from IPython.display import IFrame
IFrame('lag_proof_v2.html', width='100%', height=800)

In [ ]:
## Part 4: The Last Line of Defense

The Fed's tools are blunt, its mandate is contradictory, and its decisions work 
on a delay. Political independence doesn't fix any of that — but losing it removes 
even the limited ability to manage those constraints.

History offers a natural experiment: **Burns vs. Volcker.** Same institution, 
same mandate, different levels of independence. The outcomes couldn't be more different. 
And the pressure Powell faces today looks familiar.

In [ ]:
from IPython.display import IFrame
IFrame('era_comparison.html', width='100%', height=950)

In [ ]:
## Conclusion

The Fed is a structurally constrained institution asked to do something structurally difficult.

The dual mandate creates a contradiction that data confirms: **low inflation and low 
unemployment rarely coexist for long.** The transmission lag means even correct 
decisions produce variable outcomes. And political pressure — when it succeeds — 
breaks the institution's ability to function at all.

What would work better? The evidence points toward:
- **Rule-based frameworks** (Taylor Rule, NGDP targeting) to reduce discretionary variance
- **Better fiscal-monetary coordination** so monetary policy isn't the only tool
- **Structural independence protections** that are harder to erode politically

The Fed was never designed to fully succeed. The question is whether we keep 
asking it to — or design something better.

---
*Mark Hamer · Data Visualization · Spring 2026*  
*Data: Federal Reserve FRED API · Bureau of Labor Statistics · Case-Shiller Home Price Index · NASDAQ Composite*